#### SG2 swerve base keyboard teleop — `real`

`sim_base_controller_sg2.ipynb` 와 같은 키/속도로 실제 베이스를 움직입니다.

```
MacBook (이 노트북)  ──ZMQ :5561 {"base":[vx,vy,wz]}──▶  Orin inbound.py ──▶ /cmd_vel ──▶ swerve_drive_controller
```

**Orin 준비 (ai_worker 컨테이너)**
```bash
worker_bringup_teleop                              # 리더 없이 follower 만
SG2_ZMQ_SUB_IP=<맥북 IP> worker_inbound           # 아래 셀이 출력하는 IP. strict 모드(SG2_FIXED_QUEST=1)에선 base 무시됨
```

**안전**
- `Robot command` 가 `ON` 일 때만 전송. `OFF` 로 바꾸거나 셀을 끝내면 0 을 보내고 멈춤
- `SPACE` 즉시 정지. 전송이 끊기면 로봇이 1초 뒤 스스로 멈춤 (`cmd_vel_timeout`)
- inbound.py 가 0.3 m/s, 0.6 rad/s 로 한 번 더 제한
- 화면의 로봇 위치는 **보낸 명령을 적분한 값**(open loop)이라 실제 위치와 다를 수 있음
- 처음엔 주변을 비우고 `Speed = SLOW` 로 시작

In [2]:
import os
if "__vsc_ipynb_file__" in globals(): os.chdir(os.path.dirname(__vsc_ipynb_file__))   # VS Code 커널 cwd 보정
%run ../package/init_project.py

import numpy as np

from ri_motion_v5_package.init_scripts.init_ipython_setup import *
from ri_motion_v5_package.init_scripts.init_qt import *
from ri_motion_v5_package.mujoco_sim import *
from ri_motion_v5_package.kinematics import *
from ri_motion_v5_package.utility import *
from ri_motion_v5_package.qt import *
from utils import *
from package.base_control import *   # not `from base_control`: PACKAGE_DIR is on sys.path only during %run

app = get_qt_app()


[ffw_sg2_vr_teleoperation] initialized.
 - PROJECT_DIR: /Users/yujin/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation


In [3]:
import time

PUB_TO_SG2_PORT = 5561
PUB_TO_SG2_IP = get_ip_address()
SEND_HZ = 20
print("MacBook IP:",PUB_TO_SG2_IP,"→ Orin: SG2_ZMQ_SUB_IP=%s worker_inbound"%PUB_TO_SG2_IP)

MacBook IP: 192.168.6.101 → Orin: SG2_ZMQ_SUB_IP=192.168.6.101 worker_inbound


In [4]:
xml_path = merge_mjcfs(
    included_mjcf_files = [
        str(FLOOR_XML_PATH),
        str(SG2_XML_PATH),
    ],
    output_xml_path = str(SG2_SCENE_XML_PATH),)
env = MuJoCoParser(rel_xml_path=xml_path,verbose=False)
env.reset()
base_z = get_sg2_base_z(env)
print("steer joints:",SG2_BASE_STEER_JOINT_NAMES)
print("drive joints:",SG2_BASE_DRIVE_JOINT_NAMES)

[merge_mjcfs] Merging [2] MJCF files:
  - [0] [/Users/yujin/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/floor_simple_white.xml]
  - [1] [/Users/yujin/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/ffw_sg2.xml]
[merge_mjcfs] Saved merged XML to:[/Users/yujin/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/scene_ffw_sg2_metaquest3.xml]
MuJoCo model loaded from [/Users/yujin/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/scene_ffw_sg2_metaquest3.xml].
steer joints: ['left_wheel_steer_joint', 'right_wheel_steer_joint', 'rear_wheel_steer_joint']
drive joints: ['left_wheel_drive_joint', 'right_wheel_drive_joint', 'rear_wheel_drive_joint']


In [5]:
env.init_viewer(title="SG2 Base Teleop (real)",x_offset=0.23,y_offset=0.00,width=0.77,height=1.0,fontscale=180)
env.viewer.set_cam_info(200.0,4.0,-35.0,[0.0,0.0,0.8])

teleop = SG2BaseTeleop()
ctrl_radio = create_sg2_base_command_widget(real=True)
pub_robot = ZMQPubSub(role="pub",port=PUB_TO_SG2_PORT,ip=PUB_TO_SG2_IP,verbose=True)

[ZMQ Publisher] pid:[39353] endpoint:[tcp://0.0.0.0:5561]
[ZMQ] role:[pub] ip:[192.168.6.101] port:[5561] hwm:[1] conflate:[True]


In [6]:
hz = 30
tmr = SimpleTimer(name="Base Real",Hz=hz,verbose=False)
tmr_send = SimpleTimer(name="Base Pub",Hz=SEND_HZ,verbose=False)
tmr.start(); tmr_send.start(); env.reset_wall_time(); env.reset_sim_time()

def send_stop(n=5):
    for _ in range(n):
        pub_robot.send(obj={"base":[0.0,0.0,0.0]}); time.sleep(0.02)

prev_on_flag = False
n_sent = 0
last_sent = [0.0,0.0,0.0]
n_sent_moving = 0
try:
    while env.is_viewer_alive() and ctrl_radio.isVisible():
        app.processEvents()

        on_flag = ctrl_radio.get_button_option(label_text="Robot command") == "ON"
        if on_flag != prev_on_flag:
            teleop.stop()
            if not on_flag:
                send_stop()
            print("[Base Pub] %s"%("ON → MuJoCo 창을 클릭한 뒤 W/A/S/D/Q/E" if on_flag else "OFF (stop sent)"))
            prev_on_flag = on_flag

        if tmr.do_run():
            teleop.scale = SG2_BASE_SPEED_SCALES[ctrl_radio.get_button_option(label_text="Speed")]
            if on_flag:
                teleop.read_keys(env)
            else:
                teleop.stop()
            teleop.update(1.0/hz)
            apply_sg2_base_to_env(env,teleop,base_z)

            if ctrl_radio.get_button_option(label_text="Camera") == "FOLLOW":
                env.viewer.set_cam_info(lookat=[teleop.pose[0],teleop.pose[1],0.8])
            env.plot_global_coordinate_axes()
            draw_sg2_base(env,teleop,base_z)
            status = "ROBOT %s  sent %d (moving %d)  last [%+.2f %+.2f %+.2f]"%(
                "ON" if on_flag else "OFF",n_sent,n_sent_moving,*last_sent)
            env.viewer_text_overlay("Base",make_sg2_base_overlay_text(teleop,extra=status),loc="top left")
            env.render()
            tmr.end()

        if tmr_send.do_run():
            if on_flag:
                msg = teleop.base_msg(); pub_robot.send(obj=msg); n_sent += 1
                last_sent = msg["base"]
                if any(v != 0.0 for v in last_sent):
                    if n_sent_moving == 0: print("[Base Pub] first non-zero command sent:",last_sent)
                    n_sent_moving += 1
            tmr_send.end()
except KeyboardInterrupt:
    print("Interrupted.")
finally:
    send_stop()
    pub_robot.close()
    ctrl_radio.close()
    env.close_viewer()
    print("Done. (stop sent)")

[Base Pub] ON → MuJoCo 창을 클릭한 뒤 W/A/S/D/Q/E


2026-09-19 17:52:43.491 python[39353:2945655] TSM AdjustCapsLockLEDForKeyTransitionHandling - _ISSetPhysicalKeyboardCapsLockLED Inhibit
2026-09-19 17:52:43.494 python[39353:2945655] error messaging the mach port for IMKCFRunLoopWakeUpReliable


[Base Pub] first non-zero command sent: [0.016666666666666666, 0.0, 0.0]
Interrupted.
[ZMQ Publisher] pid:[39353] sent close signal.
[ZMQ PUB] pid:[39353] Socket closed.
Done. (stop sent)
